In [1]:
from google.colab import drive
drive.mount('/content/drive')

# === UPDATE THIS TO YOUR SHAP EXPERIMENT FOLDER ===
BASE_DIR = "/content/drive/MyDrive/longformer_runs/run_paper_v1"

DATA_DIR    = f"{BASE_DIR}/data"
RESULTS_DIR = f"{BASE_DIR}/results_longformer"
SHAP_DIR    = f"{BASE_DIR}/shap_outputs"

import os
print("DATA_DIR   :", DATA_DIR)
print("RESULTS_DIR:", RESULTS_DIR)
print("SHAP_DIR   :", SHAP_DIR)


Mounted at /content/drive
DATA_DIR   : /content/drive/MyDrive/longformer_runs/run_paper_v1/data
RESULTS_DIR: /content/drive/MyDrive/longformer_runs/run_paper_v1/results_longformer
SHAP_DIR   : /content/drive/MyDrive/longformer_runs/run_paper_v1/shap_outputs


In [2]:
import pandas as pd
import numpy as np
import os

subset_path = os.path.join(SHAP_DIR, "shap_samples_batched_with_row.csv")
shap_samples_df = pd.read_csv(subset_path)

print("Loaded SHAP subset (30 samples):")
shap_samples_df.head()


Loaded SHAP subset (30 samples):


,idx_in_test_after_cleaning,text,label,shap_row
0,295,public ArrayList<ErrorMsg> getWarnings() {...,1,0
1,298,public int atAdPos(final int pos) {\n ...,0,1
2,361,public List getAnchorHRefs(boolean duplica...,1,2
3,869,\tpublic Index parseAndUpdateIndex(List<JaxbRo...,0,3
4,915,\tprivate boolean isBreakOnOpcode(Integer opco...,0,4


In [3]:
pred_path = os.path.join(SHAP_DIR, "test_predictions.csv")

if not os.path.isfile(pred_path):
    raise FileNotFoundError("test_predictions.csv not found! You must generate predictions first.")

preds_df = pd.read_csv(pred_path)

print("Loaded predictions for full test set.")
preds_df.head()


Loaded predictions for full test set.


,idx_in_test_after_cleaning,label_test,p_class0,p_class1,pred_label
0,0,1,0.947870,0.052130,0
1,1,0,0.949378,0.050622,0
2,2,1,0.824398,0.175602,0
3,3,0,0.945457,0.054543,0
4,4,1,0.469261,0.530739,1


In [4]:
join_df = shap_samples_df.merge(
    preds_df,
    on="idx_in_test_after_cleaning",
    how="inner"
)

print("Merged subset shape:", join_df.shape)
join_df.head()


Merged subset shape: (30, 8)


,idx_in_test_after_cleaning,text,label,shap_row,label_test,p_class0,p_class1,pred_label
0,295,public ArrayList<ErrorMsg> getWarnings() {...,1,0,1,0.964182,0.035818,0
1,298,public int atAdPos(final int pos) {\n ...,0,1,0,0.901602,0.098398,0
2,361,public List getAnchorHRefs(boolean duplica...,1,2,1,0.931769,0.068231,0
3,869,\tpublic Index parseAndUpdateIndex(List<JaxbRo...,0,3,0,0.921380,0.078620,0
4,915,\tprivate boolean isBreakOnOpcode(Integer opco...,0,4,0,0.762079,0.237921,0


In [6]:
from sklearn.metrics import confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
import os

# Extract true and predicted labels from the merged SHAP subset
y_true = join_df["label_test"].to_numpy()
y_pred = join_df["pred_label"].to_numpy()

# Compute confusion matrix (same ordering: 0 → inconsistent, 1 → consistent)
cm = confusion_matrix(y_true, y_pred, labels=[0,1])

plt.figure(figsize=(7,6))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=["Pred: Inconsistent (0)", "Pred: Consistent (1)"],
    yticklabels=["True: Inconsistent (0)", "True: Consistent (1)"],
    cbar_kws={'label': 'Count'}
)

plt.title("Confusion Matrix (SHAP 30-Sample Subset)", fontsize=14)
plt.xlabel("Predicted", fontsize=12)
plt.ylabel("True", fontsize=12)
plt.tight_layout()

cm_path = os.path.join(SHAP_DIR, "cm_shap_subset_readable.png")
plt.savefig(cm_path, dpi=200)
plt.close()

print("Saved:", cm_path)


Saved: /content/drive/MyDrive/longformer_runs/run_paper_v1/shap_outputs/cm_shap_subset_readable.png
